# Chapter 8 &mdash; Error-Correcting Design II: the NFA that Counts Dings

**Concept 7 of the Chapter 8 decomposition:** *Error-Correcting Design II: the NFA that Silently Corrects and Counts Dings*

When the wrong symbol arrives, correct it silently and move to a state layer recording one more ding.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Error-Correcting-NFA/Concept-Error-Correcting-NFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The same language, designed as a **machine** rather than an expression.

Read `0101` position by position. When the **expected** symbol arrives, advance within
the current layer. When the **wrong** symbol arrives, advance *and* drop to the next
layer &mdash; the layer index records how many "dings" (corrections) have occurred.

Layers $0, 1, 2$ are accepting at the end; there is no layer 3, so a third ding kills
the token.

This is **error-correcting decoding** in miniature, and the layered-state-name idea
recurs whenever you must count a bounded resource.

## 2. Definitions

### Generate the layered NFA

In [ ]:
TARGET = '0101'
MAXD = 2

def layered_nfa(target, maxd):
    lines = ['NFA']
    n = len(target)
    def nm(i, d):
        if i == 0 and d == 0: return 'I'
        return ('F' if i == n else 'S') + '_p%d_d%d' % (i, d)
    for i, want in enumerate(target):
        other = '1' if want == '0' else '0'
        for d in range(maxd + 1):
            lines.append('%s : %s -> %s' % (nm(i, d), want, nm(i+1, d)))
            if d < maxd:
                lines.append('%s : %s -> %s   !! ding %d' % (nm(i, d), other, nm(i+1, d+1), d+1))
    return md2mc('\n'.join(lines))

N = layered_nfa(TARGET, MAXD)
print("states :", len(N["Q"]), " final :", len(N["F"]))

### The reference

In [ ]:
def ham(a, b): return sum(x != y for x, y in zip(a, b))
def within(s, k=MAXD): return len(s) == len(TARGET) and ham(s, TARGET) <= k

## 3. Tests

The layers are visible in the state names.

In [ ]:
for q in sorted(N["Q"]):
    print("  ", q)
print("\nthe _d<k> suffix IS the number of corrections made so far")

Accepts exactly the strings within distance 2.

In [ ]:
from itertools import product
acc = [''.join(p) for p in product('01', repeat=4) if accepts_nfa(N, ''.join(p))]
print("accepted (%d) :" % len(acc), acc)
assert set(acc) == {''.join(p) for p in product('01', repeat=4) if within(''.join(p))}
assert len(acc) == 11

A **third** ding has nowhere to go, so the token dies.

In [ ]:
worst = ''.join('1' if ch == '0' else '0' for ch in TARGET)   # distance 4
print("complement of the target :", worst, " distance", ham(worst, TARGET))
assert not accepts_nfa(N, worst)
three = '1011'          # 1-0, 0-1, 1-0 differ; the last symbol agrees
print("%r distance %d accepted? %s" % (three, ham(three, TARGET), accepts_nfa(N, three)))
assert ham(three, TARGET) == 3 and not accepts_nfa(N, three)
two = '1111'            # only two positions differ -- this one IS accepted
print("%r distance %d accepted? %s" % (two, ham(two, TARGET), accepts_nfa(N, two)))
assert ham(two, TARGET) == 2 and accepts_nfa(N, two)

Raising the layer count raises the tolerated distance, exactly.

In [ ]:
for k in range(0, 5):
    Nk = layered_nfa(TARGET, k)
    a = sum(1 for p in product('01', repeat=4) if accepts_nfa(Nk, ''.join(p)))
    from math import comb
    expect = sum(comb(4, j) for j in range(k+1))
    print("maxd=%d : %2d accepted, C(4,0..%d) sums to %2d" % (k, a, k, expect))
    assert a == expect

## 4. Animation

The layered machine: each row is one more correction.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Make the dinging **non**-silent: emit a marker symbol. What machine class is that?
2. How many states for a 7-bit target at distance 3?
3. Why is the layer index bounded &mdash; and what would an unbounded one require?

In [ ]:
# Your work for the exercises above.